In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import pickle

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'H:\DPJI\IDDPedestrian\annotations\gopro',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'obs_len': 15, 
    'pred_len': 45,
    'seq_len': 60,
    'hidden_size': 256,   
    'latent_dim': 64,    
    'batch_size': 128,   
    'epochs': 50,        
    'lr': 0.001,
    'best_k': 20,        
    'kl_weight': 1.0     
}
print(f"✅ BiTraP Config Loaded. Device: {CONFIG['device']}")

✅ BiTraP Config Loaded. Device: cuda


In [2]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        print("📂 Parsing XMLs...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        W, H = 1920.0, 1080.0
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                for track in root.findall('track'):
                    if track.attrib['label'] != 'pedestrian': continue
                    
                    track_data = []
                    for box in track.findall('box'):
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        
                        # Center of Bottom Edge (Feet)
                        cx = (xtl + xbr) / 2.0
                        cy = ybr 
                        w = xbr - xtl
                        h = ybr - ytl
                        
                        # Normalize 0-1
                        track_data.append([cx/W, cy/H, w/W, h/H]) 
                    
                    track_data = np.array(track_data)
                    if len(track_data) < self.seq_len: continue
                    
                    # Sliding Window
                    for i in range(0, len(track_data) - self.seq_len + 1, 15):
                        seq = track_data[i : i+self.seq_len]
                        
                        # 1. Absolute Positions
                        obs_abs = seq[:obs_len]
                        pred_abs = seq[obs_len:]
                        
                        # 2. Velocities (Differences)
                        vel_seq = np.zeros_like(seq)
                        vel_seq[1:] = seq[1:] - seq[:-1]
                        vel_seq[0] = vel_seq[1] 
                        
                        obs_vel = vel_seq[:obs_len, 0:2]
                        target_vel = vel_seq[obs_len:, 0:2]
                        
                        # 3. Relative Target (Displacement from last observation)
                        # This is crucial for Position Loss
                        last_obs = obs_abs[-1, 0:2] # [x, y]
                        target_rel = pred_abs[:, 0:2] - last_obs # Offset from start of prediction
                        
                        self.samples.append({
                            'obs_vel': obs_vel,
                            'target_vel': target_vel,
                            'target_rel': target_rel,
                            'obs_abs': obs_abs,
                            'pred_abs': pred_abs
                        })
            except: pass
                
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            torch.tensor(item['obs_vel'], dtype=torch.float32),
            torch.tensor(item['target_vel'], dtype=torch.float32),
            torch.tensor(item['target_rel'], dtype=torch.float32),
            torch.tensor(item['obs_abs'], dtype=torch.float32),
            torch.tensor(item['pred_abs'], dtype=torch.float32)
        )

dataset = IDDTrajectoryDataset(CONFIG['xml_root'], CONFIG['obs_len'], CONFIG['pred_len'])
print(f"Dataset Size: {len(dataset)}")

📂 Parsing XMLs...


  0%|          | 0/33 [00:00<?, ?it/s]

Dataset Size: 18587


In [3]:
class BiTraP_CVAE(nn.Module):
    def __init__(self, input_dim=2, hidden_size=256, latent_dim=64):
        super(BiTraP_CVAE, self).__init__()
        
        # 1. Bidirectional Past Encoder
        self.past_encoder = nn.LSTM(input_dim, hidden_size, batch_first=True, bidirectional=True)
        self.past_dim = hidden_size * 2 # Because of bidirectional
        
        # 2. Future Encoder (Training only)
        self.future_encoder = nn.LSTM(input_dim, hidden_size, batch_first=True, bidirectional=True)
        self.fut_dim = hidden_size * 2
        
        # 3. Latent Space Projection
        self.fc_mu = nn.Linear(self.past_dim + self.fut_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.past_dim + self.fut_dim, latent_dim)
        
        # Prior Network (P(z|X))
        self.fc_mu_prior = nn.Linear(self.past_dim, latent_dim)
        self.fc_logvar_prior = nn.Linear(self.past_dim, latent_dim)
        
        # 4. Decoder
        self.decoder_init = nn.Linear(latent_dim + self.past_dim, hidden_size)
        self.decoder = nn.LSTM(input_dim, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, input_dim)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, obs_vel, target_vel=None, training=True):
        # Encode Past
        _, (h_past, _) = self.past_encoder(obs_vel)
        # Concat forward and backward hidden states
        h_past = torch.cat((h_past[-2], h_past[-1]), dim=1) # [B, Hidden*2]
        
        if training and target_vel is not None:
            # Encode Future
            _, (h_fut, _) = self.future_encoder(target_vel)
            h_fut = torch.cat((h_fut[-2], h_fut[-1]), dim=1)
            
            # CVAE Posterior
            h_combined = torch.cat([h_past, h_fut], dim=1)
            mu = self.fc_mu(h_combined)
            logvar = self.fc_logvar(h_combined)
        else:
            # Prior
            mu = self.fc_mu_prior(h_past)
            logvar = self.fc_logvar_prior(h_past)
            
        z = self.reparameterize(mu, logvar)
        
        # Decode
        # Init state with Z and Past Context
        decoder_h = torch.tanh(self.decoder_init(torch.cat([z, h_past], dim=1))).unsqueeze(0)
        decoder_c = torch.zeros_like(decoder_h)
        
        outputs = []
        curr_input = obs_vel[:, -1, :].unsqueeze(1) # Start with last observed velocity
        
        pred_len = target_vel.size(1) if target_vel is not None else CONFIG['pred_len']
        
        for _ in range(pred_len):
            out, (decoder_h, decoder_c) = self.decoder(curr_input, (decoder_h, decoder_c))
            pred_vel = self.fc_out(out)
            outputs.append(pred_vel)
            curr_input = pred_vel # Autoregressive
            
        return torch.cat(outputs, dim=1), mu, logvar

model_bitrap = BiTraP_CVAE(hidden_size=CONFIG['hidden_size'], latent_dim=CONFIG['latent_dim']).to(CONFIG['device'])
print("✅ BiTraP (Bidirectional) Initialized.")

✅ BiTraP (Bidirectional) Initialized.


In [4]:
# Split
train_size = int(0.8 * len(dataset))
val_set = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))
train_set = torch.utils.data.Subset(dataset, range(0, train_size))

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

optimizer = optim.Adam(model_bitrap.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.96) # Gentle decay

print("🚀 Starting Training with Position Loss...")

for epoch in range(CONFIG['epochs']):
    model_bitrap.train()
    train_loss = 0
    
    # KL Annealing
    kl_w = min(1.0, (epoch + 1) / 15.0) * CONFIG['kl_weight']
    
    for obs_vel, target_vel, target_rel, _, _ in tqdm(train_loader, leave=False):
        obs_vel = obs_vel.to(CONFIG['device'])
        target_vel = target_vel.to(CONFIG['device'])
        target_rel = target_rel.to(CONFIG['device']) # Relative positions
        
        optimizer.zero_grad()
        
        # 1. Predict Velocities
        pred_vel, mu, logvar = model_bitrap(obs_vel, target_vel, training=True)
        
        # 2. Reconstruct Path (Integration)
        # Cumsum of velocities = Displacement from start
        pred_path = torch.cumsum(pred_vel, dim=1)
        
        # 3. Loss Calculation
        # Position Loss (Matches Drift) - Weighted heavily
        loss_pos = nn.functional.mse_loss(pred_path, target_rel, reduction='sum')
        
        # Velocity Loss (Smoothness)
        loss_vel = nn.functional.mse_loss(pred_vel, target_vel, reduction='sum')
        
        # KLD
        loss_kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        # Combined Loss
        loss = loss_pos + (loss_vel * 10) + (kl_w * loss_kld)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bitrap.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
    scheduler.step()
    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.2f} | KL W: {kl_w:.2f}")

torch.save(model_bitrap.state_dict(), "bitrap_model_refined.pth")

🚀 Starting Training with Position Loss...


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 156.54 | KL W: 0.07


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 43.28 | KL W: 0.13


...

Epoch 48 | Train Loss: 9.83 | KL W: 1.00


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 49 | Train Loss: 9.56 | KL W: 1.00


  0%|          | 0/117 [00:00<?, ?it/s]

Epoch 50 | Train Loss: 9.80 | KL W: 1.00


In [5]:
def calculate_metrics_refined(model, loader):
    model.eval()
    mse_traj_list = [] 
    cmse_list = [] 
    cfmse_list = [] 
    
    W, H = 1920.0, 1080.0
    K = CONFIG['best_k']
    
    with torch.no_grad():
        for obs_vel, _, _, obs_abs, target_abs in tqdm(loader, desc="Evaluating"):
            obs_vel = obs_vel.to(CONFIG['device'])
            
            # Ground Truth Pixels
            gt_px = target_abs[:, :, 0:2].numpy() * [W, H]
            gt_wh = target_abs[:, :, 2:4].numpy() * [W, H]
            
            last_obs_pos = obs_abs[:, -1, 0:2].to(CONFIG['device'])
            
            best_mse_batch = np.full(obs_vel.size(0), float('inf'))
            best_pred_px = np.zeros((obs_vel.size(0), CONFIG['pred_len'], 2))
            
            for _ in range(K):
                # Inference
                pred_vel, _, _ = model(obs_vel, training=False)
                
                # Integrate
                pred_path = torch.cumsum(pred_vel, dim=1) # Offset from 0
                pred_pos = last_obs_pos.unsqueeze(1) + pred_path # Absolute 0-1
                
                # Pixels
                pred_px_sample = pred_pos.cpu().numpy() * [W, H]
                
                # MSE
                mse = np.mean(np.sum((pred_px_sample - gt_px)**2, axis=2), axis=1)
                
                improved = mse < best_mse_batch
                best_mse_batch[improved] = mse[improved]
                best_pred_px[improved] = pred_px_sample[improved]

            # Store metrics
            mse_traj_list.extend(best_mse_batch)
            
            # C-MSE (Final point)
            pred_end = best_pred_px[:, -1, :]
            gt_end = gt_px[:, -1, :]
            c_mse = np.sum((pred_end - gt_end)**2, axis=1)
            cmse_list.extend(c_mse)
            
            # CF-MSE (Center + Foot)
            # Center Y = Foot Y - Height/2
            h_vec = gt_wh[:, -1, 1]
            pred_cy = pred_end[:, 1] - h_vec/2
            gt_cy = gt_end[:, 1] - h_vec/2
            
            pred_center = np.stack([pred_end[:, 0], pred_cy], axis=1)
            gt_center = np.stack([gt_end[:, 0], gt_cy], axis=1)
            
            center_err = np.sum((pred_center - gt_center)**2, axis=1)
            cfmse_list.extend(center_err + c_mse)

    return np.mean(mse_traj_list), np.mean(cmse_list), np.mean(cfmse_list)

mse, c_mse, cf_mse = calculate_metrics_refined(model_bitrap, val_loader)

print("="*40)
print(f"✅ FINAL OPTIMIZED RESULTS (Best-of-{CONFIG['best_k']}):")
print(f"   MSE (Avg):     {mse:.2f}")
print(f"   C-MSE (1.5s):  {c_mse:.2f}")
print(f"   CF-MSE (1.5s): {cf_mse:.2f}")
print("="*40)

with open('result_bitrap_final.pkl', 'wb') as f:
    pickle.dump({'MSE': mse, 'C-MSE': c_mse, 'CF-MSE': cf_mse}, f)

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

✅ FINAL OPTIMIZED RESULTS (Best-of-20):
   MSE (Avg):     4059.21
   C-MSE (1.5s):  18364.50
   CF-MSE (1.5s): 36728.99
